# Plausibilitätsprüfung des Netzwerkdatensatzes

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import pandas as pd
from freight_routing.data_loader import NetworkDataLoader
from freight_routing.data_models import TransportArcTemplate

# Physikalische Obergrenzen der Durchschnittsgeschwindigkeit je Modus [km/h]
SPEED_CEILING = {"road": 130.0, "rail": 120.0, "ship": 40.0, "air": 950.0}
MODE_ORDER = [
    "ship",
    "rail",
    "road",
    "air",
]  # erwartet aufsteigend in Kosten & Emissionen

Setup OK


In [ ]:
def check_network(nd) -> list[tuple[str, str, bool]]:
    """Führt die Plausibilitätsprüfungen aus und gibt (Aspekt, Detail, bestanden) zurück."""
    transport = [t for t in nd.arc_templates if isinstance(t, TransportArcTemplate)]
    checks = []

    # 1) Distanzen positiv
    dmin = min(t.distance for t in transport)
    checks.append(("Distanzen positiv", f"min. Distanz = {dmin:.1f} km", dmin > 0))

    # 2) Fahrzeiten positiv
    tmin = min(t.duration_min for t in transport)
    checks.append(("Fahrzeiten positiv", f"min. Dauer = {tmin} min", tmin > 0))

    # 3) Geschwindigkeit physikalisch plausibel (max. je Modus unter Obergrenze)
    speed_ok, details = True, []
    for m, ceil in SPEED_CEILING.items():
        speeds = [t.distance / (t.duration_min / 60) for t in transport if t.mode == m]
        if not speeds:
            continue
        vmax = max(speeds)
        details.append(f"{m}: {vmax:.0f}≤{ceil:.0f}")
        speed_ok &= vmax <= ceil
    checks.append(
        ("Geschwindigkeit ≤ physikal. Grenze [km/h]", ", ".join(details), speed_ok)
    )

    # 4) Kapazitäten fahrzeugtypisch und positiv
    caps = {m: nd.capacities[m] for m in MODE_ORDER}
    cap_ok = all(v > 0 for v in caps.values())
    checks.append(
        (
            "Kapazitäten > 0 [t]",
            ", ".join(f"{m}={caps[m]:.0f}" for m in MODE_ORDER),
            cap_ok,
        )
    )

    # 5) Kostenrangfolge ship < rail < road < air
    costs = [nd.mode_factors[m].cost_per_ton_km for m in MODE_ORDER]
    cost_ok = all(a < b for a, b in zip(costs, costs[1:]))
    checks.append(
        (
            "Kostenrangfolge [€/t·km]",
            " < ".join(f"{m} {c:g}" for m, c in zip(MODE_ORDER, costs)),
            cost_ok,
        )
    )

    # 6) Emissionsrangfolge ship < rail < road < air
    emis = [nd.mode_factors[m].emissions_kg_per_ton_km for m in MODE_ORDER]
    emis_ok = all(a < b for a, b in zip(emis, emis[1:]))
    checks.append(
        (
            "Emissionsrangfolge [kg CO₂/t·km]",
            " < ".join(f"{m} {e:g}" for m, e in zip(MODE_ORDER, emis)),
            emis_ok,
        )
    )

    # 7) Nichtnegativität aller Faktoren
    nonneg = all(
        f.cost_per_ton_km >= 0 and f.emissions_kg_per_ton_km >= 0
        for f in nd.mode_factors.values()
    )
    nonneg &= all(
        (t.cost is None or t.cost >= 0) and (t.emissions is None or t.emissions >= 0)
        for t in transport
    )
    checks.append(("Nichtnegativität Kosten/Emissionen", "alle Faktoren ≥ 0", nonneg))

    return checks


rows = []
for name in ("small", "medium", "large"):
    nd = NetworkDataLoader.from_json(PROJECT_ROOT / f"dataset/{name}_network.json")
    for aspect, detail, ok in check_network(nd):
        rows.append(
            {
                "Datensatz": name,
                "Prüfaspekt": aspect,
                "Detail": detail,
                "Status": "OK" if ok else "FEHLER",
            }
        )

df = pd.DataFrame(rows)
df

,Datensatz,Prüfaspekt,Detail,Status
0,small,Distanzen positiv,min. Distanz = 2.9 km,OK
1,small,Fahrzeiten positiv,min. Dauer = 5 min,OK
2,small,Geschwindigkeit ≤ physikal. Grenze [km/h],"road: 104≤130, rail: 47≤120, ship: 22≤40, air:...",OK
3,small,Kapazitäten > 0 [t],"ship=8000, rail=1000, road=40, air=50",OK
4,small,Kostenrangfolge [€/t·km],ship 0.4 < rail 0.7 < road 1.2 < air 3.5,OK
5,small,Emissionsrangfolge [kg CO₂/t·km],ship 0.015 < rail 0.025 < road 0.09 < air 0.6,OK
6,small,Nichtnegativität Kosten/Emissionen,alle Faktoren ≥ 0,OK
7,medium,Distanzen positiv,min. Distanz = 2.8 km,OK
8,medium,Fahrzeiten positiv,min. Dauer = 5 min,OK
9,medium,Geschwindigkeit ≤ physikal. Grenze [km/h],"road: 104≤130, rail: 47≤120, ship: 22≤40, air:...",OK


In [3]:
all_ok = (df["Status"] == "OK").all()
print(f"Geprüfte Regeln: {len(df)} | bestanden: {(df['Status']=='OK').sum()}")
assert all_ok, "Mindestens eine Plausibilitätsprüfung ist fehlgeschlagen!"
print("Alle Plausibilitätsprüfungen bestanden.")

Geprüfte Regeln: 21 | bestanden: 21
Alle Plausibilitätsprüfungen bestanden.
